1 — Génération des données clients fictives

In [3]:
# ════════════════════════════════════════════════════════
# Objectif : simuler une base de données d'une compagnie
# d'assurance avec 5000 clients omnicanaux
# ════════════════════════════════════════════════════════

import pandas as pd        # manipulation des données tabulaires
import numpy as np         # calculs numériques et génération aléatoire
from datetime import datetime, timedelta  # gestion des dates
import random              # génération aléatoire simple

# Fixer les graines aléatoires pour avoir des résultats
# reproductibles — même résultat à chaque exécution
random.seed(42)
np.random.seed(42)

# ── Paramètres de simulation ─────────────────────────────
N_CLIENTS = 5000  # nombre de clients à générer

# Les 5 canaux d'interaction disponibles
CANAUX = ["web", "app_mobile", "agence", "telephone", "email"]

# Les 5 produits proposés par la compagnie
PRODUITS = ["auto", "habitation", "sante", "prevoyance", "epargne"]

# Les 4 statuts possibles d'un client
STATUTS = ["prospect", "client_actif", "client_inactif", "résilié"]

# ── Génération de la table clients ──────────────────────
# Chaque ligne = un client avec ses caractéristiques
clients = pd.DataFrame({

    # Identifiant unique : CLI00001, CLI00002, etc.
    "client_id": [f"CLI{str(i).zfill(5)}" for i in range(1, N_CLIENTS+1)],

    # Âge entre 18 et 80 ans — distribution uniforme
    "age": np.random.randint(18, 80, N_CLIENTS),

    # Ancienneté en années — de 0 (nouveau) à 25 ans
    "anciennete_ans": np.random.randint(0, 25, N_CLIENTS),

    # Statut client avec probabilités réalistes :
    # 15% prospects, 55% actifs, 20% inactifs, 10% résiliés
    "statut": np.random.choice(STATUTS, N_CLIENTS, p=[0.15, 0.55, 0.20, 0.10]),

    # Nombre de produits souscrits — la plupart ont 1 ou 2 produits
    "nb_produits": np.random.choice([1,2,3,4,5], N_CLIENTS, p=[0.35,0.30,0.20,0.10,0.05]),

    # Canal principal utilisé par le client
    # Le web est dominant (35%), puis app mobile (25%)
    "canal_principal": np.random.choice(CANAUX, N_CLIENTS, p=[0.35,0.25,0.20,0.15,0.05]),

    # Région géographique du client
    "region": np.random.choice(["IDF","PACA","ARA","OCC","NAQ","BFC"], N_CLIENTS),

    # Revenu mensuel : distribution normale autour de 2800€
    # Limité entre 1200€ et 8000€ pour rester réaliste
    "revenu_mensuel": np.round(np.random.normal(2800, 900, N_CLIENTS).clip(1200, 8000), 0),
})

# ── Vérification des données générées ───────────────────
print(f"✅ {len(clients)} clients générés")
print("\nAperçu des 5 premières lignes :")
print(clients.head())
print("\nRépartition des statuts clients :")
print(clients["statut"].value_counts())

✅ 5000 clients générés

Aperçu des 5 premières lignes :
  client_id  age  anciennete_ans        statut  nb_produits canal_principal  \
0  CLI00001   56              22  client_actif            4             web   
1  CLI00002   69               9      prospect            3      app_mobile   
2  CLI00003   46               9      prospect            3          agence   
3  CLI00004   32               3       résilié            1      app_mobile   
4  CLI00005   60               7  client_actif            1             web   

  region  revenu_mensuel  
0    BFC          2521.0  
1   PACA          1950.0  
2    BFC          2971.0  
3    NAQ          3420.0  
4    IDF          3499.0  

Répartition des statuts clients :
statut
client_actif      2823
client_inactif     956
prospect           742
résilié            479
Name: count, dtype: int64


2 — Les interactions omnicanales

In [4]:
# ════════════════════════════════════════════════════════
# CELLULE 2 — Génération des interactions omnicanales
# Objectif : simuler l'historique de toutes les interactions
# entre les clients et la compagnie sur 24 mois
# Chaque ligne = 1 contact client (appel, visite, clic...)
# ════════════════════════════════════════════════════════

# ── Paramètres des interactions ─────────────────────────
DATE_DEBUT = datetime(2023, 1, 1)   # début de la période
DATE_FIN   = datetime(2024, 12, 31) # fin de la période

# Types d'interactions possibles selon le canal
TYPES_INTERACTION = {
    "web"        : ["visite_site", "devis_en_ligne", "souscription", "espace_client"],
    "app_mobile" : ["ouverture_app", "declaration_sinistre", "consultation_contrat", "paiement"],
    "agence"     : ["rdv_conseiller", "souscription", "reclamation", "renouvellement"],
    "telephone"  : ["appel_entrant", "appel_sortant", "reclamation", "resiliation"],
    "email"      : ["email_ouvert", "email_clique", "email_ignore", "reponse_email"],
}

# Résultats possibles d'une interaction
RESULTATS = ["succès", "abandon", "en_attente", "escalade"]

interactions = []  # liste qui va accumuler toutes les interactions

# ── Génération des interactions par client ───────────────
for _, client in clients.iterrows():

    # Nombre d'interactions selon le statut du client
    # Un client actif interagit plus qu'un prospect ou résilié
    if client["statut"] == "client_actif":
        nb_interactions = np.random.randint(5, 25)
    elif client["statut"] == "prospect":
        nb_interactions = np.random.randint(1, 8)
    elif client["statut"] == "client_inactif":
        nb_interactions = np.random.randint(1, 5)
    else:  # résilié
        nb_interactions = np.random.randint(1, 4)

    for _ in range(nb_interactions):

        # Date aléatoire dans la période de 24 mois
        jours_random = random.randint(0, (DATE_FIN - DATE_DEBUT).days)
        date_interaction = DATE_DEBUT + timedelta(days=jours_random)

        # Canal utilisé pour cette interaction
        # Favorise le canal principal du client (60% du temps)
        if random.random() < 0.60:
            canal = client["canal_principal"]
        else:
            canal = random.choice(CANAUX)

        # Type d'interaction correspondant au canal
        type_interaction = random.choice(TYPES_INTERACTION[canal])

        # Durée de l'interaction en minutes selon le canal
        if canal == "agence":
            duree = np.random.randint(15, 90)   # rendez-vous plus longs
        elif canal == "telephone":
            duree = np.random.randint(3, 30)    # appels téléphoniques
        else:
            duree = np.random.randint(1, 15)    # interactions digitales rapides

        # Score de satisfaction (NPS) — entre 0 et 10
        # Les agences et téléphones ont de meilleurs scores en moyenne
        if canal in ["agence", "telephone"]:
            satisfaction = min(10, max(0, int(np.random.normal(7.5, 1.5))))
        else:
            satisfaction = min(10, max(0, int(np.random.normal(6.5, 2.0))))

        interactions.append({
            "interaction_id"  : f"INT{str(len(interactions)+1).zfill(7)}",
            "client_id"       : client["client_id"],
            "date"            : date_interaction.strftime("%Y-%m-%d"),
            "canal"           : canal,
            "type_interaction": type_interaction,
            "duree_minutes"   : duree,
            "satisfaction"    : satisfaction,
            "resultat"        : np.random.choice(RESULTATS, p=[0.65, 0.20, 0.10, 0.05]),
        })

# ── Conversion en DataFrame ──────────────────────────────
interactions_df = pd.DataFrame(interactions)

# ── Vérification ─────────────────────────────────────────
print(f"✅ {len(interactions_df):,} interactions générées")
print(f"   Période : {interactions_df['date'].min()} → {interactions_df['date'].max()}")
print(f"\nAperçu :")
print(interactions_df.head())
print(f"\nRépartition par canal :")
print(interactions_df["canal"].value_counts())
print(f"\nSatisfaction moyenne par canal :")
print(interactions_df.groupby("canal")["satisfaction"].mean().round(2).sort_values(ascending=False))

✅ 47,637 interactions générées
   Période : 2023-01-01 → 2024-12-31

Aperçu :
  interaction_id client_id        date  canal type_interaction  duree_minutes  \
0     INT0000001  CLI00001  2024-10-16    web     souscription              5   
1     INT0000002  CLI00001  2023-09-08    web      visite_site             12   
2     INT0000003  CLI00001  2024-11-23  email     email_ouvert             12   
3     INT0000004  CLI00001  2024-08-27    web      visite_site              2   
4     INT0000005  CLI00001  2023-04-06    web      visite_site              3   

   satisfaction  resultat  
0             8   abandon  
1             3    succès  
2             4    succès  
3             9   abandon  
4             5  escalade  

Répartition par canal :
canal
web           13233
app_mobile    11283
agence         9678
telephone      8046
email          5397
Name: count, dtype: int64

Satisfaction moyenne par canal :
canal
agence        7.01
telephone     7.00
web           6.01
app_mobile   

3 — Les contrats souscrits

In [5]:
# ════════════════════════════════════════════════════════
# CELLULE 3 — Génération des contrats souscrits
# Objectif : simuler le portefeuille de contrats d'assurance
# Chaque ligne = 1 contrat souscrit par un client
# ════════════════════════════════════════════════════════

# Prix annuels moyens par produit (en euros)
# Ces montants sont réalistes pour le marché français
PRIX_MOYENS = {
    "auto"       : 650,
    "habitation" : 280,
    "sante"      : 1200,
    "prevoyance" : 450,
    "epargne"    : 2000,
}

# Statuts possibles d'un contrat
STATUTS_CONTRAT = ["actif", "suspendu", "résilié", "en_renouvellement"]

contrats = []  # liste vide qui va accumuler tous les contrats

# ── Génération des contrats pour chaque client ──────────
for _, client in clients.iterrows():

    # Chaque client a autant de contrats que son nb_produits
    nb_contrats = client["nb_produits"]

    # On choisit des produits différents pour ce client
    # sans doublon (un client ne souscrit pas 2x le même produit)
    produits_client = random.sample(PRODUITS, min(nb_contrats, len(PRODUITS)))

    for produit in produits_client:

        # Date de souscription calculée selon l'ancienneté du client
        # Un client avec 10 ans d'ancienneté a souscrit il y a ~10 ans
        jours_anciennete = client["anciennete_ans"] * 365
        jours_souscription = random.randint(0, max(1, jours_anciennete))
        date_souscription = datetime(2024, 12, 31) - timedelta(days=jours_souscription)

        # Prime annuelle = prix moyen avec variation de ±20%
        # Simule les différences de profil (jeune conducteur = plus cher)
        variation = np.random.uniform(0.80, 1.20)
        prime = round(PRIX_MOYENS[produit] * variation, 2)

        # Statut du contrat cohérent avec le statut du client
        if client["statut"] == "résilié":
            # Un client résilié → tous ses contrats sont résiliés
            statut_contrat = "résilié"
        elif client["statut"] == "client_inactif":
            # Un client inactif → souvent suspendu ou en attente
            statut_contrat = np.random.choice(
                STATUTS_CONTRAT, p=[0.40, 0.35, 0.15, 0.10]
            )
        else:
            # Un client actif → majorité de contrats actifs
            statut_contrat = np.random.choice(
                STATUTS_CONTRAT, p=[0.80, 0.05, 0.05, 0.10]
            )

        contrats.append({
            # Identifiant unique du contrat
            "contrat_id"       : f"CTR{str(len(contrats)+1).zfill(6)}",
            "client_id"        : client["client_id"],
            "produit"          : produit,
            "date_souscription": date_souscription.strftime("%Y-%m-%d"),
            "prime_annuelle"   : prime,
            "statut_contrat"   : statut_contrat,
            # Nombre de sinistres déclarés — la plupart n'en ont pas
            "nb_sinistres"     : np.random.choice(
                [0, 1, 2, 3, 4], p=[0.55, 0.25, 0.12, 0.05, 0.03]
            ),
        })

# ── Conversion en DataFrame ──────────────────────────────
contrats_df = pd.DataFrame(contrats)

# ── Vérification ─────────────────────────────────────────
print(f"✅ {len(contrats_df):,} contrats générés")
print(f"\nAperçu :")
print(contrats_df.head())
print(f"\nRépartition par produit :")
print(contrats_df["produit"].value_counts())
print(f"\nPrime annuelle moyenne par produit :")
print(contrats_df.groupby("produit")["prime_annuelle"].mean().round(2).sort_values(ascending=False))
print(f"\nStatuts des contrats :")
print(contrats_df["statut_contrat"].value_counts())

✅ 11,024 contrats générés

Aperçu :
  contrat_id client_id     produit date_souscription  prime_annuelle  \
0  CTR000001  CLI00001        auto        2023-11-15          694.93   
1  CTR000002  CLI00001       sante        2016-03-11         1369.20   
2  CTR000003  CLI00001     epargne        2003-10-01         2180.73   
3  CTR000004  CLI00001  prevoyance        2007-01-31          415.28   
4  CTR000005  CLI00002        auto        2019-02-26          591.63   

      statut_contrat  nb_sinistres  
0              actif             1  
1              actif             0  
2              actif             1  
3              actif             0  
4  en_renouvellement             2  

Répartition par produit :
produit
auto          2250
habitation    2246
sante         2195
prevoyance    2180
epargne       2153
Name: count, dtype: int64

Prime annuelle moyenne par produit :
produit
epargne       1996.80
sante         1198.61
auto           653.41
prevoyance     451.28
habitation     280.

4 — Les sinistres déclarés 

In [6]:
# ════════════════════════════════════════════════════════
# CELLULE 4 — Génération des sinistres déclarés
# Objectif : simuler les déclarations de sinistres
# Chaque ligne = 1 sinistre déclaré sur un contrat
# Un sinistre = un événement couvert (accident, vol, maladie...)
# ════════════════════════════════════════════════════════

# Types de sinistres possibles par produit
# Chaque produit a ses propres types d'événements couverts
TYPES_SINISTRES = {
    "auto"       : ["accident", "vol", "bris_de_glace", "incendie", "vandalisme"],
    "habitation" : ["dégât_des_eaux", "vol", "incendie", "catastrophe_naturelle"],
    "sante"      : ["hospitalisation", "consultation", "optique", "dentaire"],
    "prevoyance" : ["arrêt_travail", "invalidité", "décès"],
    "epargne"    : ["rachat_partiel", "rachat_total"],
}

# Statuts possibles d'un sinistre
STATUTS_SINISTRE = ["déclaré", "en_instruction", "indemnisé", "refusé", "clôturé"]

sinistres = []  # liste vide pour accumuler les sinistres

# ── Génération des sinistres pour chaque contrat ─────────
for _, contrat in contrats_df.iterrows():

    # On ne génère des sinistres que si le contrat en a
    # (nb_sinistres est déjà défini dans la table contrats)
    nb = contrat["nb_sinistres"]

    if nb == 0:
        # Pas de sinistre pour ce contrat → on passe
        continue

    for _ in range(nb):

        # Date du sinistre : après la souscription du contrat
        date_souscription = datetime.strptime(
            contrat["date_souscription"], "%Y-%m-%d"
        )
        # Le sinistre se produit entre la souscription et fin 2024
        jours_disponibles = (datetime(2024, 12, 31) - date_souscription).days
        if jours_disponibles <= 0:
            continue
        date_sinistre = date_souscription + timedelta(
            days=random.randint(0, jours_disponibles)
        )

        # Montant indemnisé selon le type de produit
        # Ces fourchettes sont réalistes pour le marché français
        produit = contrat["produit"]
        if produit == "auto":
            montant = round(np.random.uniform(500, 8000), 2)
        elif produit == "habitation":
            montant = round(np.random.uniform(300, 15000), 2)
        elif produit == "sante":
            montant = round(np.random.uniform(50, 3000), 2)
        elif produit == "prevoyance":
            montant = round(np.random.uniform(1000, 20000), 2)
        else:  # epargne
            montant = round(np.random.uniform(5000, 50000), 2)

        # Durée de traitement en jours
        # Les sinistres complexes (invalidité, incendie) prennent plus de temps
        if produit in ["prevoyance", "habitation"]:
            duree_traitement = np.random.randint(15, 120)
        else:
            duree_traitement = np.random.randint(3, 45)

        sinistres.append({
            # Identifiant unique du sinistre
            "sinistre_id"      : f"SIN{str(len(sinistres)+1).zfill(6)}",
            "contrat_id"       : contrat["contrat_id"],
            "client_id"        : contrat["client_id"],
            "produit"          : produit,
            "date_sinistre"    : date_sinistre.strftime("%Y-%m-%d"),
            "type_sinistre"    : random.choice(TYPES_SINISTRES[produit]),
            "montant_indemnise": montant,
            "duree_traitement" : duree_traitement,
            # Statut du sinistre — la majorité sont clôturés
            "statut_sinistre"  : np.random.choice(
                STATUTS_SINISTRE, p=[0.05, 0.10, 0.50, 0.05, 0.30]
            ),
        })

# ── Conversion en DataFrame ───────────────────────────────
sinistres_df = pd.DataFrame(sinistres)

# ── Vérification ──────────────────────────────────────────
print(f"✅ {len(sinistres_df):,} sinistres générés")
print(f"\nAperçu :")
print(sinistres_df.head())
print(f"\nRépartition par produit :")
print(sinistres_df["produit"].value_counts())
print(f"\nMontant moyen indemnisé par produit :")
print(sinistres_df.groupby("produit")["montant_indemnise"].mean().round(2).sort_values(ascending=False))
print(f"\nDurée moyenne de traitement par produit (jours) :")
print(sinistres_df.groupby("produit")["duree_traitement"].mean().round(1).sort_values(ascending=False))

✅ 8,214 sinistres générés

Aperçu :
  sinistre_id contrat_id client_id  produit date_sinistre   type_sinistre  \
0   SIN000001  CTR000001  CLI00001     auto    2024-02-08             vol   
1   SIN000002  CTR000003  CLI00001  epargne    2015-05-18    rachat_total   
2   SIN000003  CTR000005  CLI00002     auto    2019-05-09   bris_de_glace   
3   SIN000004  CTR000005  CLI00002     auto    2021-11-18   bris_de_glace   
4   SIN000005  CTR000006  CLI00002  epargne    2024-04-19  rachat_partiel   

   montant_indemnise  duree_traitement statut_sinistre  
0            6904.22                19          refusé  
1            9049.17                13         clôturé  
2            6441.48                20       indemnisé  
3            4065.05                22       indemnisé  
4           13556.53                37         clôturé  

Répartition par produit :
produit
auto          1673
sante         1666
prevoyance    1663
habitation    1637
epargne       1575
Name: count, dtype: int64

Mo

5 — Sauvegarde les 4 tables

In [7]:
# ════════════════════════════════════════════════════════
# CELLULE 5 — Sauvegarde des 4 tables en CSV
# Objectif : persister les données générées dans le dossier
# data/ pour les réutiliser dans tous les prochains notebooks
# ════════════════════════════════════════════════════════

import os

# Créer le dossier data/ s'il n'existe pas déjà
os.makedirs("data", exist_ok=True)

# ── Sauvegarde des 4 tables ──────────────────────────────
clients.to_csv("data/clients.csv", index=False)
interactions_df.to_csv("data/interactions.csv", index=False)
contrats_df.to_csv("data/contrats.csv", index=False)
sinistres_df.to_csv("data/sinistres.csv", index=False)

# ── Résumé final du dataset ──────────────────────────────
print("✅ Toutes les tables sauvegardées dans data/")
print()
print("═" * 45)
print("  RÉSUMÉ DU DATASET ASSURANCE DATA 360")
print("═" * 45)
print(f"  Clients       : {len(clients):>8,} lignes")
print(f"  Interactions  : {len(interactions_df):>8,} lignes")
print(f"  Contrats      : {len(contrats_df):>8,} lignes")
print(f"  Sinistres     : {len(sinistres_df):>8,} lignes")
print("─" * 45)
total = len(clients) + len(interactions_df) + len(contrats_df) + len(sinistres_df)
print(f"  TOTAL         : {total:>8,} lignes de données")
print("═" * 45)
print()
print("Fichiers créés :")
for f in os.listdir("data"):
    taille = os.path.getsize(f"data/{f}") / 1024
    print(f"  data/{f:<30} {taille:.1f} Ko")

✅ Toutes les tables sauvegardées dans data/

═════════════════════════════════════════════
  RÉSUMÉ DU DATASET ASSURANCE DATA 360
═════════════════════════════════════════════
  Clients       :    5,000 lignes
  Interactions  :   47,637 lignes
  Contrats      :   11,024 lignes
  Sinistres     :    8,214 lignes
─────────────────────────────────────────────
  TOTAL         :   71,875 lignes de données
═════════════════════════════════════════════

Fichiers créés :
  data/clients.csv                    233.0 Ko
  data/interactions.csv               3049.0 Ko
  data/sinistres.csv                  658.7 Ko
  data/contrats.csv                   597.3 Ko
